## Installing Dependency 

In [1]:
!pip install -q \
langchain \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
langchain-groq \
langchain-google-genai \
langchain-openai \
langchain-core \
faiss-cpu \
pypdf \
sentence-transformers \
transformers \
torch \
huggingface_hub \
groq \
langsmith \
python-dotenv \
tiktoken 


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [28]:
import os
from dotenv import load_dotenv

load_dotenv()
LLM_PROVIDER = "Groq"
LLM_MODEL = "openai/gpt-oss-20b"
LLM_API_KEY = os.getenv("GROQ_API_KEY")

if not LLM_API_KEY:
    raise ValueError("GROQ_API_KEY is not set.")


In [4]:
import os 
from langchain_community.document_loaders import PyPDFDirectoryLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
CORPUS_PATH = "./zyro-dynamics-hr-corpus/"
loader = PyPDFDirectoryLoader(CORPUS_PATH) 
documents = loader.load() 
print(f"Loaded {len(documents)} documents")  

Loaded 39 documents


In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100) 
chunks = splitter.split_documents(documents) 
print(f"Created {len(chunks)} chunks") 


Created 107 chunks


# Hugging Face Embedding 

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings 
model = HuggingFaceEmbeddings(
    model_name ='sentence-transformers/all-MiniLM-L6-v2' 
    
)
print("Embeddings Done") 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3431.81it/s]


Embeddings Done


# VectorDB + Retrival
FAISS Vector DB

In [10]:
from langchain_community.vectorstores import FAISS 
vectorstore = FAISS.from_documents(chunks, model) 

retriever = vectorstore.as_retriever(search_kwargs={"k":3}) 
print("done") 

done


# LLM Initialization
LLM model -  groq

In [29]:
from langchain_groq import ChatGroq
model = ChatGroq(
model = LLM_MODEL, 
temperature = 0.7,
max_tokens = 500 ,
groq_api_key=LLM_API_KEY
)

print(f"LLM Model {model} is ready") 

LLM Model metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0'}} profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True} client=<groq.resources.chat.completions.Completions object at 0x00000154C212DF50> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000154C21AF550> model_name='openai/gpt-oss-20b' model_kwargs={} groq_api_key=SecretStr('**********') max_tokens=500 is ready


# Rag Chain


In [31]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable 

RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are a HR assistant. Use the following context to answer the question at the end. If you don't know the answer, just say you don't know. Do not try to make up an answer.
      
      Context:
      {context}
      
      Question: {question}
      Answer:"""
)

def format_docs(docs):
    return ".\n\n".join(d.page_content for d in docs) 

@traceable(name = "rag_chain") 
def rag_chain(question : str):
    docs = retriever.invoke(question) 
    context  = format_docs(docs)
    chain = RAG_PROMPT | model | StrOutputParser()
    answer = chain.invoke({"context":context , "question":question}) 
    return {"answer":answer,"sources": docs}  

# Guardial_prompt 

In [36]:
GUARDIAL_PROMPT = ChatPromptTemplate.from_template(""" 
You are a scope classifier for the Zyro Dynamics HR assistant.

Decide whether the employee's question is about Zyro Dynamics internal HR topics,
including leave, attendance, holidays, payroll, benefits, reimbursements, travel
or expense policies, onboarding, workplace conduct, employment procedures, or
other information likely covered by the company HR documents.

Return IN_SCOPE if it is an HR-related Zyro Dynamics question, even when the
available documents may not contain the answer.

Return OUT_OF_SCOPE if it is unrelated to Zyro Dynamics HR, such as general
knowledge, coding, mathematics, news, entertainment, personal advice, casual
conversation, questions about another company, or requests for private employee
information.

Respond with exactly one word: IN_SCOPE or OUT_OF_SCOPE.

Question: {question}
""")

REFUSAL_MESSAGE = (
    "I'm the Zyro Dynamics HR assistant and can help only with questions about "
    "company HR policies and procedures, such as leave, attendance, benefits, "
    "reimbursements, workplace conduct, and onboarding. Please ask an HR-related question."
)

def ask_bot(question: str):
    guardial_chain = GUARDIAL_PROMPT | model | StrOutputParser()
    verdict = guardial_chain.invoke({"question": question}).strip().upper() 
    if "OUT_OF_SCOPE" in verdict:
        return{"answer": REFUSAL_MESSAGE, "sources": []} 
    return rag_chain(question) 
print("Zyro Dynamics HR assistant is ready. Ask your question!")


Zyro Dynamics HR assistant is ready. Ask your question!


In [43]:
import pandas as pd 
test_df = pd.read_csv("test.csv") 
test_df 
answers = [] 
for i , row in test_df.iterrows():
    q =row["question"] 
    result = ask_bot(q) 
    answers.append(result["answer"]) 
    print(f"[{i+1}/{len(test_df)}]{q}\n-> {result['answer']}\n{'-'*60}") 

test_df["answer"] = answers

submission = test_df[["question_id" , "answer"]] 
submission.to_csv("submission.csv" , index=False) 
submission.head() 

[1/20]How does my Earned Leave accrue every month?
-> Earned Leave (EL) accrues at a fixed rate each month:

* **During probation:** 0.5 days per month (available only after probation confirmation).  
* **After probation (post‑probation):** 1.25 days per month.  

(Employees become eligible for the full 15 days of EL after completing one year of continuous service with at least 240 days worked that year.)
------------------------------------------------------------
[2/20]How much Earned Leave can I carry forward to next year?
-> I don’t know.
------------------------------------------------------------
[3/20]How many weeks of Maternity Leave am I entitled to?
-> You are entitled to:

- **26 weeks** of paid Maternity Leave for your first and second live births.  
- **12 weeks** of paid Maternity Leave for a third live birth.  

Additionally, you may take up to **8 weeks of pre‑natal leave** before the expected delivery date.
------------------------------------------------------------
[

,question_id,answer
0,Q01,Earned Leave (EL) accrues at a fixed rate each...
1,Q02,I don’t know.
2,Q03,You are entitled to:\n\n- **26 weeks** of paid...
3,Q04,"Yes. According to the policy, a medical certif..."
4,Q05,Your salary is credited to your registered ban...
